# Домашнее задание 3 — Агент-аналитик системных промптов (MCP + Skills)

### Задания

1. **Файловые инструменты агента** — реализовать инструменты для сохранения/чтения результатов анализа
2. **Пайплайн: агенты с tool loop** — реализовать агента и оркестратор пайплайна

### Критерии оценки (10 баллов)

| # | Критерий | Балл |
|---|----------|------|
| 1 | Реализованы все 5 файловых инструментов (`save_analysis`, `list_analyses`, `read_analysis`, `save_report`, `read_report`) + tool schemas + `LOCAL_TOOLS` | 1 |
| 2 | Написаны все 4 skill-файла (`skills.md`, `prompt_analysis.md`, `analysis_report.md`, `prompt_generator.md`) с двухуровневой структурой (Level 1 — метаданные в `skills.md`, Level 2 — полная процедура) | 1 |
| 3 | `prompt_analysis.md` содержит ≥5 критериев оценки промпта (Role/Persona, Few-shot, Chain-of-thought, Output format, Guardrails и т.д.) | 1 |
| 4 | Реализован `run_agent` с корректным tool loop: маршрутизация вызовов (MCP / `read_skill` / локальные), трассировка через `AgentTrace`, чистый контекст на каждый вызов | 1 |
| 5 | Сценарий `prompt_analysis` работает: агент загружает skill → читает файл из GitHub → анализирует → сохраняет через `save_analysis` | 1 |
| 6 | Сценарий `analysis_report` работает: агент читает все анализы → формирует сводный отчёт → сохраняет через `save_report` | 1 |
| 7 | Сценарий `prompt_generator` работает: агент читает отчёт → генерирует новый системный промпт → сохраняет результат | 1 |
| 8 | Progressive disclosure реализован как для skills, так и для MCP-инструментов | 1 |
| 9–10 | Сценарий `prompt_analysis` реализован через субагента (отдельный агентный вызов для каждого файла, а не один монолитный контекст) | 2 |

---
## 0. Подготовка

In [17]:
import warnings

warnings.filterwarnings("ignore")

import sys
import json
import asyncio
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import nest_asyncio

nest_asyncio.apply()

from src.config import settings
from openai import OpenAI

In [18]:
client = OpenAI(
    base_url="https://polza.ai/api/v1",
    api_key="pza_4U5FlDhFI2Z21-qHKB1K7pJBWmbwNheJ",
)

# Многошаговый tool loop с вложенным progressive disclosure требует сильной модели.
# Мелкие модели срываются: забывают read_skill и шлют невалидный JSON в аргументах.
# Локальный llama.cpp отдаёт ту модель, что загружена, и имя отсюда игнорирует —
# оно остаётся только как метка в логах.
MODEL = "deepseek/deepseek-v4-flash"

# Окно контекста задаётся при старте сервера (-c), а не моделью: llama.cpp
# режет запрос по n_ctx и отвечает 400 exceed_context_size_error. Система + схемы
# инструментов занимают ~3.7k токенов ещё до полезной нагрузки.
#
# На 8 ГБ VRAM (RTX 3070) Ministral-3-8B Q5_K_M занимает 5.6 ГиБ весами, поэтому
# на KV-кэш остаётся ~1.5 ГиБ: 32k f16 (4.5 ГиБ) не влезает, рабочий вариант —
#   llama-server -m <модель> -c 16384 -np 1 -fa on -ctk q8_0 -ctv q8_0 --jinja
# (-fa обязателен для квантованного KV; -np делит окно между слотами).
MIN_CONTEXT = 16_384  # ниже этого пайплайн не проходит даже с урезанными бюджетами


def check_context_window(base_url: str = "http://127.0.0.1:8080") -> int:
    """Вернуть n_ctx локального сервера (0, если это не llama.cpp)."""
    import httpx

    try:
        meta = httpx.get(f"{base_url}/v1/models", timeout=5).json()["data"][0]
    except Exception as ex:  # облачный API (polza и т.п.) — проверять нечего
        print(f"\u2139\ufe0f Проверка n_ctx пропущена: {type(ex).__name__}")
        return 0

    n_ctx = int(meta.get("meta", {}).get("n_ctx", 0))
    print(f"\U0001f9e0 Модель на сервере: {Path(meta['id']).name}")
    print(f"\U0001f4cf Окно контекста: n_ctx={n_ctx}")
    if 0 < n_ctx < MIN_CONTEXT:
        print(f"\u26a0\ufe0f Мало: нужно минимум {MIN_CONTEXT}, см. команду запуска в комментарии выше.")
    return n_ctx


SERVER_CTX = check_context_window() or 128_000  # облачные модели считаем просторными


def check_server_slots(base_url: str = "http://127.0.0.1:8080") -> int:
    """Сколько запросов сервер считает одновременными (llama.cpp: -np). 0 — не llama.cpp."""
    import httpx

    try:
        return int(httpx.get(f"{base_url}/props", timeout=5).json().get("total_slots", 0))
    except Exception:
        return 0


SERVER_SLOTS = check_server_slots()
print(f"\U0001f9f5 Слотов на сервере: {SERVER_SLOTS or 'нет данных (облачный API)'}")

# Бюджеты пайплайна считаем от фактического окна, а не константами под одну машину:
# сервер может подняться с любым -c, и ноутбук должен пережить это без правок.
CHARS_PER_TOKEN = 3.5  # смесь русского и английского в markdown


def ctx_chars(share: float) -> int:
    """Доля окна контекста, выраженная в символах."""
    return int(SERVER_CTX * share * CHARS_PER_TOKEN)


print(f"\U0001f4b0 Бюджет одного наблюдения: {ctx_chars(0.25)} символов")


ℹ️ Проверка n_ctx пропущена: ReadTimeout
🧵 Слотов на сервере: нет данных (облачный API)
💰 Бюджет одного наблюдения: 112000 символов


---
## 0.1 Утилиты для трассировки агента

Код из практики — необходим для выполнения заданий.

In [19]:
# ============================================================
# Утилиты для трассировки агента (из практики 3)
# ============================================================
from dataclasses import dataclass, field


def _extract_usage(response) -> dict:
    """Извлечь токены из response.usage."""
    usage = getattr(response, "usage", None)
    if not usage:
        return {"prompt_tokens": 0, "cached_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "context_size": 0}

    prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
    completion_tokens = getattr(usage, "completion_tokens", 0) or 0
    total_tokens = getattr(usage, "total_tokens", 0) or 0

    cached_tokens = getattr(usage, "prompt_cache_hit_tokens", 0) or 0
    if not cached_tokens:
        details = getattr(usage, "prompt_tokens_details", None)
        if details:
            cached_tokens = getattr(details, "cached_tokens", 0) or 0

    context_size = prompt_tokens + cached_tokens
    return {
        "prompt_tokens": prompt_tokens,
        "cached_tokens": cached_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "context_size": context_size,
    }


@dataclass
class AgentStep:
    """Один шаг агента — для трассировки."""

    step_number: int
    tool_calls: list[dict] = field(default_factory=list)
    observations: list[dict] = field(default_factory=list)
    prompt_tokens: int = 0
    cached_tokens: int = 0
    context_size: int = 0
    context_size_delta: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    final_answer: str | None = None


@dataclass
class AgentTrace:
    """Полный лог работы агента."""

    question: str
    steps: list[AgentStep] = field(default_factory=list)
    total_steps: int = 0
    prompt_tokens_total: int = 0
    completion_tokens_total: int = 0
    total_tokens_total: int = 0
    latest_context_size: int = 0
    max_context_size: int = 0
    final_answer: str = ""

    def summary(self) -> str:
        lines = [
            f"Вопрос: {self.question}",
            f"Шагов: {self.total_steps}",
            f"Prompt tokens (биллинг): {self.prompt_tokens_total}",
            f"Completion tokens: {self.completion_tokens_total}",
            f"Всего tokens: {self.total_tokens_total}",
            "",
        ]
        for s in self.steps:
            lines.append(f"--- Шаг {s.step_number} ---")
            if s.total_tokens:
                lines.append(
                    f"  \U0001f522 context={s.context_size} (\u0394 {s.context_size_delta:+}), "
                    f"completion={s.completion_tokens}"
                )
            for tc in s.tool_calls:
                lines.append(f"  \U0001f527 {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)[:80]})")
            for obs in s.observations:
                text = str(obs)
                lines.append(f"  \U0001f4e1 {text[:120]}{'...' if len(text) > 120 else ''}")
            if s.final_answer:
                lines.append(f"  \U0001f4ac {s.final_answer[:300]}")
        lines.append(f"\n{'=' * 50}\n\U0001f3af Финальный ответ:\n{self.final_answer[:800]}")
        return "\n".join(lines)

---
## 0.2 Источник данных — GitHub MCP Server

Подключаемся к официальному GitHub MCP-серверу для доступа к репозиторию с системными промптами.

Репозиторий [`asgeirtj/system_prompts_leaks`](https://github.com/asgeirtj/system_prompts_leaks) — коллекция реальных системных промптов (ChatGPT, Claude, Gemini, Codex и др.)

### Предварительная настройка

#### 1. Node.js и npx

**npx** — утилита из экосистемы Node.js для запуска npm-пакетов без установки.

Проверить установку:
```bash
node --version   # должно быть v18+ 
npx --version    # должно быть 9+
```

#### 2. GitHub Personal Access Token

Создаётся в [Settings → Developer settings → Personal access tokens → Tokens (classic)](https://github.com/settings/tokens).

Минимальные права: `public_repo`.

Добавьте токен в файл `.env` в корне проекта:
```
GITHUB_TOKEN=ghp_ваш_токен_здесь
```

In [20]:
# ============================================================
# Подключение к GitHub MCP Server через stdio
# ============================================================
import os
import shutil

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Windows: PATH читается процессом один раз при старте. Если Node поставили
# после запуска VS Code, npx не найдётся. Добавляем путь ДО создания параметров,
# иначе в env уйдёт снимок старого PATH.
_NODE_DIR = r"C:\Program Files\nodejs"
if os.path.isdir(_NODE_DIR) and _NODE_DIR not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + _NODE_DIR

print("npx:", shutil.which("npx") or "НЕ НАЙДЕН — перезапусти VS Code")

# Параметры подключения — запускаем официальный GitHub MCP-сервер
github_server_params = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-github"],
    env={
        **os.environ,
        "GITHUB_PERSONAL_ACCESS_TOKEN": settings.github_token,
    },
)

npx: C:\Program Files\nodejs\npx.CMD


In [21]:
# ============================================================
# Discovery: подключаемся к GitHub MCP и узнаём его возможности
# ============================================================


async def discover_github_server():
    """Подключиться к GitHub MCP-серверу и получить список инструментов."""
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("\u2705 Подключение к GitHub MCP установлено!\n")

                # Discovery: Tools
                tools_response = await session.list_tools()
                print(f"\U0001f527 Инструменты ({len(tools_response.tools)}):")
                print("-" * 55)
                for tool in tools_response.tools:
                    desc = (tool.description or "")[:80]
                    print(f"  \u2022 {tool.name}")
                    print(f"    {desc}...")
                    print()

                return tools_response.tools


github_tools = asyncio.get_event_loop().run_until_complete(discover_github_server())

✅ Подключение к GitHub MCP установлено!

🔧 Инструменты (26):
-------------------------------------------------------
  • create_or_update_file
    Create or update a single file in a GitHub repository...

  • search_repositories
    Search for GitHub repositories...

  • create_repository
    Create a new GitHub repository in your account...

  • get_file_contents
    Get the contents of a file or directory from a GitHub repository...

  • push_files
    Push multiple files to a GitHub repository in a single commit...

  • create_issue
    Create a new issue in a GitHub repository...

  • create_pull_request
    Create a new pull request in a GitHub repository...

  • fork_repository
    Fork a GitHub repository to your account or specified organization...

  • create_branch
    Create a new branch in a GitHub repository...

  • list_commits
    Get list of commits of a branch in a GitHub repository...

  • list_issues
    List issues in a GitHub repository with filtering options...

 

---
## 0.3 Skills — процедурные навыки агента

В практике мы создали skills для космического ассистента.  
Здесь используем **три навыка для анализа промптов**:

| Навык | Файл | Назначение |
|-------|------|------------|
| `prompt_analysis` | `hw_3_skills/prompt_analysis.md` | Пошаговый разбор системного промпта: секции, persona, приёмы, оценки |
| `analysis_report` | `hw_3_skills/analysis_report.md` | Формирование сводного отчёта с таблицами, цитатами и рекомендациями |
| `prompt_generator` | `hw_3_skills/prompt_generator.md` | Генерация нового промпта на основе выявленных паттернов |

### Skill Chaining

В практике skills были **независимые**. Здесь они образуют **цепочку** (chain):

```
prompt_analysis ──(структурированный разбор)──→ analysis_report
                                                       │
                                              (отчёт + рекомендации)
                                                       │
                                                       ▼
                                               prompt_generator
```

**Выход одного skill = вход следующего.** Агент сам решает, когда переключиться.

In [22]:
# ============================================================
# Реестр Skills — загружаем все skill-файлы из hw_3_skills/
# ============================================================

from pydantic import BaseModel, Field


class SkillInfo(BaseModel):
    """Метаданные одного skill-файла."""

    name: str = Field(description="имя файла (без .md)")
    title: str = Field(description="Заголовок skill")
    trigger_description: str = Field(description="Когда активировать (из секции 'Когда активировать')")
    content: str = Field(description="Полный текст skill")


def load_skills(skills_directory: Path) -> dict[str, SkillInfo]:
    """Загрузить все skill-файлы из директории (кроме skills.md)."""
    skills = {}

    for skill_file in sorted(skills_directory.glob("*.md")):
        if skill_file.name == "skills.md":
            continue

        with open(skill_file, "r", encoding="utf-8") as f:
            content = f.read()

        lines = content.split("\n")
        title = lines[0].replace("# Skill: ", "").strip() if lines else skill_file.stem

        trigger = ""
        in_trigger = False
        for line in lines:
            if "Когда активировать" in line:
                in_trigger = True
                continue
            if in_trigger:
                if line.startswith("##") and "Когда" not in line:
                    break
                trigger += line + "\n"

        name = skill_file.stem
        skills[name] = SkillInfo(
            name=name,
            title=title,
            trigger_description=trigger.strip(),
            content=content,
        )

    return skills


def load_skills_metadata(skills_directory: Path) -> str:
    """Загрузить skills.md — Level 1 метаданные для system prompt."""
    skills_md = skills_directory / "skills.md"
    if skills_md.exists():
        with open(skills_md, "r", encoding="utf-8") as f:
            return f.read()
    return "\u26a0\ufe0f Файл skills.md не найден в директории skills."


# --- Загрузка ---
skills_dir = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_skills"
SKILLS = load_skills(skills_dir)

print(f"\U0001f4c2 Директория skills: {skills_dir}")
print(f"\U0001f4cb Загружено skills: {len(SKILLS)}\n")

for name, info in SKILLS.items():
    print(f"  \u2022 {name}: {info.title}")
    print(f"    Триггер: {info.trigger_description[:80]}...")
    print()

assert len(SKILLS) == 3, "Ожидались 3 skill-файла в hw_3_skills/"

📂 Директория skills: e:\практики\hse-agent-system\lectures\lecture_3\hw_3_skills
📋 Загружено skills: 3

  • analysis_report: Формирование аналитического отчёта
    Триггер: Этот skill активируется, когда:
- Анализ отдельных промптов (навык `prompt_analy...

  • prompt_analysis: Анализ системного промпта
    Триггер: Этот skill активируется, когда нужно:
- Разобрать один системный промпт на соста...

  • prompt_generator: Генерация системного промпта
    Триггер: Этот skill активируется, когда:
- Сводный отчёт (навык `analysis_report`) уже сф...



In [23]:
# ============================================================
# Level 1: загружаем skills.md — краткий обзор навыков для system prompt
# ============================================================

skills_metadata = load_skills_metadata(skills_dir)
print("Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):\n")
print(skills_metadata)

Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):

# Skills — Навыки агента-аналитика системных промптов

Набор skills, определяющих **как** агент должен действовать при анализе системных промптов, формировании сводных отчётов и генерации новых промптов.

## Список навыков

| Навык | Файл | Краткое описание |
|-------|------|------------------|
| Анализ промпта | [prompt_analysis.md](prompt_analysis.md) | Разбор ОДНОГО системного промпта: метаданные, секции, persona, приёмы prompt engineering, оценка по 6 критериям. Результат сохраняется через `save_analysis` |
| Аналитический отчёт | [analysis_report.md](analysis_report.md) | Агрегация всех сохранённых анализов: матрица приёмов, общие паттерны, лучшие практики, сравнительные оценки. Результат сохраняется через `save_report` |
| Генерация промпта | [prompt_generator.md](prompt_generator.md) | Создание нового системного промпта для заданной предметной области на основе лучших практик из отчёта. Результат сохраняет

In [24]:
# ============================================================
# Инструмент read_skill — загрузка процедуры по требованию
# ============================================================

READ_SKILL_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "read_skill",
        "description": (
            "Загрузить детальную процедуру (skill) для решения задачи определённого типа. "
            "Вызывай этот инструмент, когда задача пользователя соответствует одному из "
            "доступных skills. После загрузки — строго следуй процедуре шаг за шагом."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": ("Имя skill для загрузки: " + ", ".join(SKILLS.keys())),
                }
            },
            "required": ["name"],
        },
    },
}


def read_skill(name: str) -> str:
    """Загрузить полный контент skill по имени."""
    if name in SKILLS:
        skill = SKILLS[name]
        return (
            f"\u2550\u2550\u2550 SKILL ЗАГРУЖЕН: {skill.title} \u2550\u2550\u2550\n\n"
            f"СТРОГО следуй этой процедуре шаг за шагом.\n\n"
            f"{skill.content}"
        )
    available = ", ".join(SKILLS.keys())
    return f"Skill '{name}' не найден. Доступные skills: {available}"


# --- Тест ---
print("\U0001f527 READ_SKILL_TOOL_SCHEMA:")
print(json.dumps(READ_SKILL_TOOL_SCHEMA, ensure_ascii=False, indent=2)[:500])
print("\n--- Тест: read_skill('prompt_analysis') ---")
print(read_skill("prompt_analysis")[:400] + "...")

🔧 READ_SKILL_TOOL_SCHEMA:
{
  "type": "function",
  "function": {
    "name": "read_skill",
    "description": "Загрузить детальную процедуру (skill) для решения задачи определённого типа. Вызывай этот инструмент, когда задача пользователя соответствует одному из доступных skills. После загрузки — строго следуй процедуре шаг за шагом.",
    "parameters": {
      "type": "object",
      "properties": {
        "name": {
          "type": "string",
          "description": "Имя skill для загрузки: analysis_report, prompt_a

--- Тест: read_skill('prompt_analysis') ---
═══ SKILL ЗАГРУЖЕН: Анализ системного промпта ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Анализ системного промпта

## Требуемые инструменты
- `list_github_tools` / `load_github_tool` / `call_github_tool` — получение текста промпта из репозитория
- `save_analysis` — сохранение результата

## Когда активировать
Этот skill активируется, когда нужно:
- Разобрать один системный промпт н...


---
## 0.4 Вспомогательные функции пайплайна

Код из практики — утилиты для обхода репозитория и конвертации MCP-инструментов.

In [25]:

def format_mcp_tools(mcp_tools) -> list[dict]:
    """Конвертировать MCP tools в OpenAI function-calling формат."""
    formatted = []
    for tool in mcp_tools:
        schema = tool.inputSchema if tool.inputSchema else {"type": "object", "properties": {}}
        formatted.append(
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description or "",
                    "parameters": schema,
                },
            }
        )
    return formatted


---
## Задание 1: Файловые инструменты агента

Агент работает в несколько этапов, и между этапами ему нужно **сохранять и читать результаты** через файловую систему.

### Что нужно реализовать

| Инструмент | Назначение | Когда используется |
|------------|-----------|--------------------|
| `save_analysis(name, content)` | Сохранить анализ одного промпта в файл | После анализа каждого промпта |
| `list_analyses()` | Получить список всех сохранённых анализов | Перед формированием отчёта |
| `read_analysis(name)` | Прочитать конкретный анализ | При формировании отчёта |
| `save_report(name, content)` | Сохранить сводный отчёт / промпт | После формирования отчёта / промпта |
| `read_report(name)` | Прочитать сохранённый отчёт | Перед генерацией промпта |

### Файловая структура

```
hw_3_output/
  analyses/       ← save_analysis / read_analysis / list_analyses
    OpenAI_Codex.md
    Anthropic_Claude.md
    ...
  reports/         ← save_report / read_report
    analysis_report.md
    generated_prompt.md
```

### Подсказки
- Не забудьте добавить tool schemas в формате OpenAI function-calling
- Соберите все локальные инструменты в маппинг `LOCAL_TOOLS`

In [26]:
# ============================================================
# Инструменты агента
# ============================================================
import re
from pathlib import PurePosixPath

# --- Директории для результатов ---
ANALYSES_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "analyses"
REPORTS_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "reports"

ANALYSES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\U0001f4c2 Директория анализов: {ANALYSES_DIR}")
print(f"\U0001f4c2 Директория отчётов:  {REPORTS_DIR}")


# ---- Вспомогательные функции ----
def _safe_name(name: str) -> str:
    """'OpenAI/Codex.md' -> 'OpenAI_Codex'. Защита от слешей, '..' и запрещённых символов."""
    name = (name or "").strip()
    if name.lower().endswith(".md"):
        name = name[:-3]
    name = re.sub(r"[\\/]+", "_", name)
    name = re.sub(r"[^\w\-. ]", "_", name)  # \w включает кириллицу
    return name.strip(" .") or "unnamed"


def _available(directory: Path) -> str:
    names = sorted(p.stem for p in directory.glob("*.md"))
    return ", ".join(names) if names else "нет файлов"


def _resolve(directory: Path, name: str) -> Path | None:
    """Найти файл по имени так, как его мог назвать агент.

    list_* печатает p.stem, а _safe_name срезает '.md'. Из-за этого имя вида
    'OpenAI_Test.md' не находилось и агент зацикливался на одном вызове.
    Пробуем нормализованное имя, затем имя как есть; выход за пределы директории отсекаем.
    """
    raw = (name or "").strip()
    candidates = [f"{_safe_name(raw)}.md"]
    if raw:
        candidates += [f"{raw}.md", raw]  # имя как есть и с досуффиксом: 'X.md' -> 'X.md.md'
    for candidate in candidates:
        path = directory / candidate
        try:
            inside = path.resolve().parent == directory.resolve()
        except OSError:
            continue
        if inside and path.is_file():
            return path
    return None


# ---- save_analysis ----
def save_analysis(name: str, content: str) -> str:
    """Сохранить сводку по анализу одного промпта."""
    path = ANALYSES_DIR / f"{_safe_name(name)}.md"
    path.write_text(content, encoding="utf-8")
    return f"\u2705 Анализ сохранён: {path.name} ({len(content)} символов)"


# ---- list_analyses ----
def list_analyses() -> str:
    """Получить список всех сохранённых анализов."""
    files = sorted(ANALYSES_DIR.glob("*.md"))
    if not files:
        return "\u274c Нет сохранённых анализов. Сначала выполни анализ промптов (skill prompt_analysis)."
    lines = [f"\U0001f4cb Сохранённых анализов: {len(files)}"]
    for p in files:
        lines.append(f"- {p.stem} ({p.stat().st_size / 1024:.1f} КБ)")
    return "\n".join(lines)


# ---- read_analysis ----
def read_analysis(name: str) -> str:
    """Прочитать сохранённый анализ промпта по имени."""
    path = _resolve(ANALYSES_DIR, name)
    if path is None:
        return (
            f"\u274c Анализ '{name}' не найден. Не повторяй вызов с тем же именем. "
            f"Доступные: {_available(ANALYSES_DIR)}"
        )
    content = path.read_text(encoding="utf-8")
    return f"\u2550\u2550\u2550 АНАЛИЗ: {path.stem} \u2550\u2550\u2550\n\n{content}"


# ---- save_report ----
def save_report(name: str, content: str) -> str:
    """Сохранить отчёт или сгенерированный промпт."""
    path = REPORTS_DIR / f"{_safe_name(name)}.md"
    path.write_text(content, encoding="utf-8")
    return f"\u2705 Отчёт сохранён: {path.name} ({len(content)} символов)"


# ---- read_report ----
def read_report(name: str) -> str:
    """Прочитать сохранённый отчёт."""
    path = _resolve(REPORTS_DIR, name)
    if path is None:
        return (
            f"\u274c Отчёт '{name}' не найден. Не повторяй вызов с тем же именем. "
            f"Доступные: {_available(REPORTS_DIR)}"
        )
    return path.read_text(encoding="utf-8")


# ---- List files in github repo ----
async def list_repo_files(session) -> list[dict]:
    """
    Обойти репозиторий через MCP и собрать список промпт-файлов.
    Возвращает [{"path": "OpenAI/Codex.md", "name": "Codex.md", "folder": "OpenAI", "size": ...}]

    Обход детерминирован, поэтому выполняется обычным Python, без участия LLM.
    """
    # Параметры репозитория
    REPO = {"owner": "asgeirtj", "repo": "system_prompts_leaks"}
    PROMPT_EXTENSIONS = {".md", ".txt"}
    SKIP_FILES = {"readme.md", "license", "license.md", "contributing.md"}

    files: list[dict] = []
    queue: list[str] = [""]  # "" = корень репозитория

    while queue:
        path = queue.pop(0)
        result = await session.call_tool("get_file_contents", {**REPO, "path": path})
        text = "\n".join(getattr(c, "text", "") for c in result.content)

        if result.isError:
            print(f"\u26a0\ufe0f get_file_contents('{path}'): {text[:200]}")
            continue

        try:
            items = json.loads(text)
        except json.JSONDecodeError:
            print(f"\u26a0\ufe0f '{path}': ответ не JSON: {text[:200]}")
            continue

        if not isinstance(items, list):  # path оказался файлом, а не папкой
            continue

        for item in items:
            if item.get("type") == "dir":
                queue.append(item["path"])
            elif item.get("type") == "file":
                p = PurePosixPath(item["path"])
                if (
                    p.suffix.lower() in PROMPT_EXTENSIONS
                    and p.name.lower() not in SKIP_FILES
                    and len(p.parts) > 1  # файлы в корне репозитория — служебные
                ):
                    files.append(
                        {
                            "path": item["path"],
                            "name": item["name"],
                            "folder": str(p.parent),
                            "size": item.get("size", 0),
                        }
                    )

    return sorted(files, key=lambda f: f["path"])


# --- Проверка файловых инструментов без LLM ---
print("\n--- Самопроверка ---")
print(save_analysis("OpenAI/Test.md", "# Тест \u2705\nтекст"))
print(read_analysis("OpenAI_Test")[:60], "...")
print(read_analysis("Нет_Такого"))
print(save_report("test_report", "отчёт"))
print(read_report("нет_такого"))
print("_safe_name('../../secret') ->", _safe_name("../../secret"))

# Уборка тестовых файлов, иначе агент примет их за настоящие анализы.
# Чистим и артефакты старых прогонов (двойное .md от прежней версии _safe_name).
for _junk in ("OpenAI_Test.md", "OpenAI_Test.md.md", "Test.md", "Test_Prompt.md"):
    (ANALYSES_DIR / _junk).unlink(missing_ok=True)
(REPORTS_DIR / "test_report.md").unlink(missing_ok=True)
print("Анализов в каталоге после уборки:", len(list(ANALYSES_DIR.glob("*.md"))))


📂 Директория анализов: e:\практики\hse-agent-system\lectures\lecture_3\hw_3_output\analyses
📂 Директория отчётов:  e:\практики\hse-agent-system\lectures\lecture_3\hw_3_output\reports

--- Самопроверка ---
✅ Анализ сохранён: OpenAI_Test.md (14 символов)
═══ АНАЛИЗ: OpenAI_Test ═══

# Тест ✅
текст ...
❌ Анализ 'Нет_Такого' не найден. Не повторяй вызов с тем же именем. Доступные: Anthropic_anthropic-interviewer, Anthropic_claude-code_agents_Explore, Anthropic_claude-code_archive_glob-tool, Anthropic_claude-code_claude-code-docs-assistant, Anthropic_claude-code_commands_btw, Anthropic_claude-code_output-styles_concise, Anthropic_claude-code_skills_artifact-capabilities_SKILL, OpenAI_Test
✅ Отчёт сохранён: test_report.md (5 символов)
❌ Отчёт 'нет_такого' не найден. Не повторяй вызов с тем же именем. Доступные: analysis_report, generated_prompt, test_report
_safe_name('../../secret') -> _.._secret
Анализов в каталоге после уборки: 7


In [27]:
# Описание каждого инструмента в формате OpenAI function-calling.

_NAME_HINT = "Имя без расширения, формат Папка_Файл, например 'OpenAI_Codex'"

FILE_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "save_analysis",
            "description": (
                "Сохранить результат анализа ОДНОГО системного промпта в markdown-файл. "
                "Вызывай после завершения анализа по процедуре prompt_analysis."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": _NAME_HINT},
                    "content": {"type": "string", "description": "Полный текст анализа в markdown"},
                },
                "required": ["name", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_analyses",
            "description": (
                "Получить список всех сохранённых анализов промптов с размерами. "
                "Вызывай перед формированием сводного отчёта."
            ),
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_analysis",
            "description": "Прочитать сохранённый анализ одного промпта. Имена бери из list_analyses.",
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string", "description": _NAME_HINT}},
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "save_report",
            "description": (
                "Сохранить сводный отчёт или сгенерированный системный промпт в markdown-файл. "
                "Для отчёта используй имя 'analysis_report', для промпта — 'generated_prompt'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя файла без расширения"},
                    "content": {"type": "string", "description": "Полный текст в markdown"},
                },
                "required": ["name", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_report",
            "description": (
                "Прочитать сохранённый отчёт. Сводный отчёт называется 'analysis_report', "
                "сгенерированный промпт — 'generated_prompt'. Если файла нет, вернётся список доступных."
            ),
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string", "description": "Имя файла без расширения"}},
                "required": ["name"],
            },
        },
    },
]


# ============================================================
# Маппинг локальных инструментов
# ============================================================

LOCAL_TOOLS = {
    "read_skill": read_skill,
    "save_analysis": save_analysis,
    "list_analyses": list_analyses,
    "read_analysis": read_analysis,
    "save_report": save_report,
    "read_report": read_report,
}

LOCAL_TOOL_SCHEMAS = [READ_SKILL_TOOL_SCHEMA, *FILE_TOOL_SCHEMAS]

print(f"\n\U0001f527 Локальные инструменты агента ({len(LOCAL_TOOLS)}):")
for name in LOCAL_TOOLS:
    print(f"  \u2022 {name}")
print(f"\U0001f4dc Схем локальных инструментов: {len(LOCAL_TOOL_SCHEMAS)}")


🔧 Локальные инструменты агента (6):
  • read_skill
  • save_analysis
  • list_analyses
  • read_analysis
  • save_report
  • read_report
📜 Схем локальных инструментов: 6


---
## Пайплайн — универсальный агент с tool loop

Реализуйте **одного универсального агента** `run_agent()`, который выполняет разные задачи в зависимости от запроса (`Начни анализ`, `сформируй сводку`, `сгенерируй промпт для такой то задачи`). 
### Подсказки

- За основу tool loop возьмите `run_reactive_agent` из практики 2 (или аналог из практики 3)
- Используйте `AgentTrace` для отслеживания шагов и токенов

In [28]:
# ============================================================
# Progressive disclosure для MCP-инструментов (критерий 8)
# ============================================================
# GitHub MCP отдаёт больше двух десятков инструментов. Класть все схемы
# в каждый запрос дорого, поэтому в контексте постоянно живут только три
# мета-инструмента, а полная схема подгружается по требованию —
# та же двухуровневая схема, что у skills.
import base64
import functools
import inspect

MCP_META_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "list_github_tools",
            "description": (
                "Показать список доступных инструментов GitHub MCP-сервера с кратким описанием. "
                "Вызывай, когда нужно получить данные из репозитория, но неизвестно, каким инструментом."
            ),
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "load_github_tool",
            "description": (
                "Загрузить полную схему параметров одного инструмента GitHub MCP. "
                "Вызывай перед первым обращением к нему через call_github_tool."
            ),
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string", "description": "Имя инструмента из list_github_tools"}},
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "call_github_tool",
            "description": (
                "Вызвать инструмент GitHub MCP-сервера. Схему параметров сначала получи "
                "через load_github_tool."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя инструмента"},
                    "arguments": {
                        "type": "string",
                        "description": 'Аргументы в виде JSON-объекта, например {"owner": "asgeirtj", "repo": "system_prompts_leaks", "path": "OpenAI/Codex.md"}',
                    },
                },
                "required": ["name", "arguments"],
            },
        },
    },
]


def _mcp_text(result) -> str:
    """Склеить текстовые блоки из ответа MCP call_tool."""
    return "\n".join(getattr(c, "text", "") for c in result.content)


def _decode_file_payload(text: str) -> str:
    """get_file_contents может вернуть содержимое файла в base64 — раскодировать."""
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        return text
    if isinstance(payload, dict) and payload.get("encoding") == "base64" and payload.get("content"):
        try:
            return base64.b64decode(payload["content"]).decode("utf-8", errors="replace")
        except Exception:
            return text
    return text


def make_mcp_meta_tools(session, mcp_tools) -> dict:
    """Собрать мета-инструменты поверх MCP-сессии."""
    by_name = {t.name: t for t in mcp_tools}

    def list_github_tools() -> str:
        lines = [f"\U0001f527 Инструменты GitHub MCP ({len(by_name)}):"]
        for name, tool in by_name.items():
            desc = (tool.description or "").split("\n")[0][:100]
            lines.append(f"- {name}: {desc}")
        lines.append("\nПолную схему получи через load_github_tool(name).")
        return "\n".join(lines)

    def load_github_tool(name: str) -> str:
        tool = by_name.get(name)
        if tool is None:
            return f"\u274c Инструмент '{name}' не найден. Доступные: {', '.join(by_name)}"
        schema = tool.inputSchema or {"type": "object", "properties": {}}
        return (
            f"\u2550\u2550\u2550 СХЕМА: {name} \u2550\u2550\u2550\n"
            f"{tool.description or ''}\n\n"
            f"Параметры:\n{json.dumps(schema, ensure_ascii=False, indent=2)}"
        )

    async def call_github_tool(name: str, arguments: str = "{}") -> str:
        if name not in by_name:
            return f"\u274c Инструмент '{name}' не найден. Доступные: {', '.join(by_name)}"
        try:
            args = json.loads(arguments) if isinstance(arguments, str) else (arguments or {})
        except json.JSONDecodeError as ex:
            return f"\u274c Аргументы не разобрались как JSON: {ex}. Передай корректный JSON-объект."
        if not isinstance(args, dict):
            return "\u274c Аргументы должны быть JSON-объектом."

        try:
            result = await session.call_tool(name, args)
        except Exception as ex:
            return f"\u274c Ошибка вызова '{name}': {type(ex).__name__}: {ex}"

        text = _mcp_text(result)
        if result.isError:
            return f"\u274c {name}: {text[:500]}"
        return _decode_file_payload(text)

    return {
        "list_github_tools": list_github_tools,
        "load_github_tool": load_github_tool,
        "call_github_tool": call_github_tool,
    }


# ============================================================
# Универсальный агент с tool loop
# ============================================================

# Одно наблюдение не должно съедать окно целиком: в диалог обязаны поместиться
# система со схемами (~3.7k токенов), текст навыка (~2.5k) и несколько наблюдений.
MAX_OBSERVATION_CHARS = ctx_chars(0.25)

BASE_SYSTEM_PROMPT = """Ты — агент-аналитик системных промптов.

Твои возможности:
1. SKILLS — процедурные навыки. Ниже список доступных навыков (Level 1).
   Прежде чем выполнять задачу, вызови read_skill и строго следуй загруженной процедуре.
2. GITHUB MCP — доступ к репозиторию с системными промптами.
   Список инструментов: list_github_tools, схема: load_github_tool, вызов: call_github_tool.
3. ФАЙЛЫ — сохранение и чтение результатов:
   save_analysis, list_analyses, read_analysis, save_report, read_report.

Правила:
- Сначала skill, потом действия.
- Не выдумывай содержимое промптов: бери его только из инструментов.
- Задача не выполнена, пока результат не сохранён в файл.
- Закончив, коротко отчитайся: что сделал и в какой файл сохранил.

{skills_metadata}"""


async def run_agent(
    question: str,
    session,
    mcp_tools,
    skills_metadata: str = "",
    extra_system: str = "",
    max_steps: int = 10,
    verbose: bool = True,
    temperature: float = 0.2,
) -> AgentTrace:
    """Универсальный агент: skills + GitHub MCP + файловые инструменты.

    Контекст собирается заново на каждый вызов — состояние живёт только в файлах.
    """
    meta_tools = make_mcp_meta_tools(session, mcp_tools)
    tool_functions = {**LOCAL_TOOLS, **meta_tools}
    tools_schema = [*LOCAL_TOOL_SCHEMAS, *MCP_META_TOOL_SCHEMAS]

    system_prompt = BASE_SYSTEM_PROMPT.format(skills_metadata=skills_metadata)
    if extra_system:
        system_prompt += f"\n\n{extra_system}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    trace = AgentTrace(question=question)
    prev_context = 0
    call_counter: dict[str, int] = {}  # сколько раз агент уже звал инструмент с теми же аргументами

    if verbose:
        print(f"[State] Задача: {question[:200]}")

    for step_num in range(1, max_steps + 1):
        if verbose:
            print(f"\n{'=' * 50}\n--- Шаг {step_num} ---")

        # Синхронный клиент внутри async: без to_thread event loop блокируется
        # и asyncio.gather не даст никакой параллельности субагентам.
        response = await asyncio.to_thread(
            client.chat.completions.create,
            model=MODEL,
            messages=messages,
            tools=tools_schema,
            temperature=temperature,
        )

        msg = response.choices[0].message
        messages.append(msg)

        usage = _extract_usage(response)
        step = AgentStep(
            step_number=step_num,
            prompt_tokens=usage["prompt_tokens"],
            cached_tokens=usage["cached_tokens"],
            context_size=usage["context_size"],
            context_size_delta=usage["context_size"] - prev_context,
            completion_tokens=usage["completion_tokens"],
            total_tokens=usage["total_tokens"],
        )
        prev_context = usage["context_size"]

        trace.prompt_tokens_total += usage["prompt_tokens"]
        trace.completion_tokens_total += usage["completion_tokens"]
        trace.total_tokens_total += usage["total_tokens"]
        trace.latest_context_size = usage["context_size"]
        trace.max_context_size = max(trace.max_context_size, usage["context_size"])

        if verbose:
            print(
                f"  \U0001f522 context={step.context_size} (\u0394 {step.context_size_delta:+}), "
                f"completion={step.completion_tokens}"
            )

        # --- финальный ответ ---
        if not msg.tool_calls:
            step.final_answer = msg.content
            trace.steps.append(step)
            trace.final_answer = msg.content or ""
            trace.total_steps = step_num
            if verbose:
                print(f"  \U0001f4ac {(msg.content or '')[:300]}")
            break

        # --- исполнение инструментов: MCP / read_skill / локальные ---
        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            step.tool_calls.append({"name": name, "args": args})
            if verbose:
                print(f"  \U0001f527 {name}({json.dumps(args, ensure_ascii=False)[:120]})")

            # Слабая модель умеет застревать: один и тот же неудачный вызов по кругу.
            # После двух повторов отвечаем отказом вместо реального исполнения.
            call_key = f"{name}:{json.dumps(args, ensure_ascii=False, sort_keys=True)}"
            call_counter[call_key] = call_counter.get(call_key, 0) + 1

            func = tool_functions.get(name)
            if call_counter[call_key] > 2:
                observation = (
                    f"\u26a0\ufe0f Вызов {name} с этими аргументами повторяется "
                    f"{call_counter[call_key]}-й раз, результат не изменится. "
                    "Не повторяй его: измени аргументы или переходи к следующему шагу процедуры."
                )
            elif func is None:
                observation = f"\u274c Неизвестный инструмент: {name}. Доступные: {', '.join(tool_functions)}"
            else:
                try:
                    if inspect.iscoroutinefunction(func):
                        observation = await func(**args)
                    else:
                        observation = await asyncio.to_thread(functools.partial(func, **args))
                except TypeError as ex:
                    observation = f"\u274c Неверные аргументы для {name}: {ex}"
                except Exception as ex:
                    observation = f"\u274c Ошибка в {name}: {type(ex).__name__}: {ex}"

            observation = str(observation)
            if len(observation) > MAX_OBSERVATION_CHARS:
                observation = (
                    observation[:MAX_OBSERVATION_CHARS] + f"\n\n[... обрезано, всего {len(observation)} символов]"
                )

            step.observations.append(observation)
            if verbose:
                print(f"  \U0001f4e1 {observation[:150]}{'...' if len(observation) > 150 else ''}")

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": observation})

        trace.steps.append(step)
    else:
        trace.total_steps = max_steps
        trace.final_answer = "\u26a0\ufe0f Достигнут лимит шагов без финального ответа."

    return trace

In [29]:
# ============================================================
# Оркестратор: сессия MCP + субагенты (критерии 9–10)
# ============================================================
# Каждый файл промпта анализирует ОТДЕЛЬНЫЙ агентный вызов со своим чистым
# контекстом. Монолитный контекст на все файлы не влез бы и мешал бы модели.
# Связь между этапами — только через файлы в hw_3_output/.

# Файл, который не влезает в наблюдение, всё равно будет обрезан — не берём такие.
MAX_PROMPT_SIZE = MAX_OBSERVATION_CHARS
# На этапе отчёта агент держит в контексте все анализы сразу, по ~3.5 КБ каждый.
MAX_PROMPTS = max(3, min(8, ctx_chars(0.35) // 3_500))
# Воркеров ровно столько, сколько слотов у сервера: лишние субагенты не ускорят
# работу, а встанут в очередь внутри llama.cpp. Облачный API (слотов не сообщает)
# держит 4 параллельных запроса спокойно.
CONCURRENCY = max(1, min(4, SERVER_SLOTS or 4))

REPO_INFO = {"owner": "asgeirtj", "repo": "system_prompts_leaks"}

print(f"Бюджеты: промпт \u2264 {MAX_PROMPT_SIZE} симв., файлов {MAX_PROMPTS}, параллельно {CONCURRENCY}")


async def with_github(work):
    """Открыть MCP-сессию один раз и выполнить в ней работу.

    Сессию держит оркестратор, а не run_agent: каждый stdio_client поднимает
    отдельный процесс npx, и на десяток субагентов это десяток процессов.
    """
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                tools = (await session.list_tools()).tools
                return await work(session, tools)


def select_prompt_files(files: list[dict]) -> list[dict]:
    """Отобрать файлы для анализа: не слишком большие, по одному на вендора сначала."""
    candidates = [f for f in files if 0 < f["size"] <= MAX_PROMPT_SIZE]

    by_folder: dict[str, list[dict]] = {}
    for f in candidates:
        by_folder.setdefault(f["folder"], []).append(f)

    selected: list[dict] = []
    # первый проход — по одному файлу от каждого вендора, чтобы отчёт был разнообразным
    for folder in sorted(by_folder):
        selected.append(by_folder[folder][0])
    # второй проход — добираем остальные
    for folder in sorted(by_folder):
        selected.extend(by_folder[folder][1:])

    return selected[:MAX_PROMPTS]


async def analyze_one_prompt(session, mcp_tools, file_info: dict) -> AgentTrace:
    """Субагент: отдельный агентный вызов на ОДИН файл промпта."""
    target_name = _safe_name(file_info["path"])
    question = (
        f"Проанализируй системный промпт из репозитория "
        f"{REPO_INFO['owner']}/{REPO_INFO['repo']}.\n"
        f"Путь к файлу: {file_info['path']}\n"
        f"Сохрани результат через save_analysis с именем '{target_name}'."
    )
    return await run_agent(
        question=question,
        session=session,
        mcp_tools=mcp_tools,
        skills_metadata=skills_metadata,
        max_steps=8,
        verbose=False,
    )


async def analyze_all_prompts(session, mcp_tools, files: list[dict]) -> list[AgentTrace]:
    """Запустить субагентов параллельно, ограничив число одновременных вызовов."""
    semaphore = asyncio.Semaphore(CONCURRENCY)

    async def guarded(file_info: dict) -> AgentTrace:
        async with semaphore:
            print(f"  \u25b6 {file_info['path']} ({file_info['size'] / 1024:.1f} КБ)")
            try:
                trace = await analyze_one_prompt(session, mcp_tools, file_info)
                print(f"  \u2714 {file_info['path']}: шагов {trace.total_steps}, токенов {trace.total_tokens_total}")
                return trace
            except Exception as ex:
                print(f"  \u2716 {file_info['path']}: {type(ex).__name__}: {ex}")
                failed = AgentTrace(question=file_info["path"])
                failed.final_answer = f"Ошибка: {ex}"
                return failed

    return await asyncio.gather(*(guarded(f) for f in files))

Бюджеты: промпт ≤ 112000 симв., файлов 8, параллельно 4


In [30]:
# ============================================================
# Тест кейс 1. Запрос на начало анализа
# ============================================================
# Сценарий prompt_analysis через субагентов: обход репозитория обычным Python,
# затем по отдельному агентному вызову на каждый файл.


async def stage_analysis(session, mcp_tools):
    files = await list_repo_files(session)
    print(f"\U0001f4c1 Найдено файлов в репозитории: {len(files)}")

    selected = select_prompt_files(files)
    print(f"\U0001f3af Отобрано для анализа: {len(selected)}\n")

    traces = await analyze_all_prompts(session, mcp_tools, selected)
    return selected, traces


selected_files, analysis_traces = asyncio.get_event_loop().run_until_complete(with_github(stage_analysis))

print(f"\n{'=' * 50}")
print(f"Субагентов отработало: {len(analysis_traces)}")
print(f"Суммарно токенов: {sum(t.total_tokens_total for t in analysis_traces)}")
print(f"Максимальный контекст одного субагента: {max((t.max_context_size for t in analysis_traces), default=0)}")
print(f"\n{list_analyses()}")

# Подробная трассировка первого субагента
if analysis_traces:
    print(f"\n{'=' * 50}\nТрассировка первого субагента:\n")
    print(analysis_traces[0].summary())

KeyboardInterrupt: 

📁 Найдено файлов в репозитории: 411
🎯 Отобрано для анализа: 8

  ▶ Anthropic/anthropic-interviewer.md (11.4 КБ)
  ▶ Anthropic/claude-code/claude-code-docs-assistant.md (18.2 КБ)
  ▶ Anthropic/claude-code/agents/Explore.md (4.0 КБ)
  ▶ Anthropic/claude-code/archive/glob-tool.md (1.2 КБ)
  ✔ Anthropic/claude-code/agents/Explore.md: шагов 5, токенов 26142
  ▶ Anthropic/claude-code/commands/btw.md (1.0 КБ)
  ✔ Anthropic/anthropic-interviewer.md: шагов 5, токенов 29366
  ▶ Anthropic/claude-code/output-styles/concise.md (1.6 КБ)
  ✔ Anthropic/claude-code/archive/glob-tool.md: шагов 6, токенов 28888
  ▶ Anthropic/claude-code/prompts/advisor-tool.md (7.7 КБ)
  ✔ Anthropic/claude-code/claude-code-docs-assistant.md: шагов 5, токенов 34647
  ▶ Anthropic/claude-code/skills/artifact-capabilities/SKILL.md (13.6 КБ)
  ✔ Anthropic/claude-code/output-styles/concise.md: шагов 6, токенов 28807
  ✔ Anthropic/claude-code/commands/btw.md: шагов 6, токенов 28384
  ✔ Anthropic/claude-code/prompts/advisor-tool

In [ ]:
# ============================================================
# Тест кейс 2. Запрос на формирование итоговой сводки
# ============================================================
# Тот же run_agent, другой запрос. Контекст чистый: входные данные агент
# берёт из файлов, сохранённых субагентами на предыдущем этапе.


async def stage_report(session, mcp_tools):
    return await run_agent(
        question=(
            "Сформируй сводный аналитический отчёт по всем сохранённым анализам "
            "системных промптов и сохрани его как 'analysis_report'."
        ),
        session=session,
        mcp_tools=mcp_tools,
        skills_metadata=skills_metadata,
        max_steps=14,
        verbose=True,
    )


report_trace = asyncio.get_event_loop().run_until_complete(with_github(stage_report))

print(f"\n{'=' * 50}")
print(report_trace.summary())
print(f"\n{'=' * 50}\nОтчёт на диске:\n")
print(read_report("analysis_report")[:2000])

[State] Задача: Сформируй сводный аналитический отчёт по всем сохранённым анализам системных промптов и сохрани его как 'analysis_report'.

--- Шаг 1 ---
  🔢 context=2021 (Δ +2021), completion=109
  🔧 read_skill({"name": "analysis_report"})
  📡 ═══ SKILL ЗАГРУЖЕН: Формирование аналитического отчёта ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Формирование аналитического отчёта

#...

--- Шаг 2 ---
  🔢 context=4895 (Δ +2874), completion=49
  🔧 list_analyses({})
  📡 📋 Сохранённых анализов: 7
- Anthropic_anthropic-interviewer (5.8 КБ)
- Anthropic_claude-code_agents_Explore (5.5 КБ)
- Anthropic_claude-code_archive_g...

--- Шаг 3 ---
  🔢 context=6360 (Δ +1465), completion=343
  🔧 read_analysis({"name": "Anthropic_anthropic-interviewer"})
  📡 ═══ АНАЛИЗ: Anthropic_anthropic-interviewer ═══

# Анализ: Anthropic / anthropic-interviewer

## Метаданные
- **Путь:** Anthropic/anthropic-interviewe...
  🔧 read_analysis({"name": "Anthropic_claude-code_agents_Explore"})
  📡 ═══ АНАЛИЗ: A

In [ ]:
# ============================================================
# Тест кейс 3. Запрос на формирование промпта для случайного ассистента
# ============================================================
# Замыкаем цепочку: prompt_analysis -> analysis_report -> prompt_generator.

TARGET_ASSISTANT = (
    "ассистент-помощник для студентов технического вуза: помогает разбирать "
    "учебные задачи по программированию, объясняет ошибки в коде и готовит к защите проектов"
)


async def stage_generator(session, mcp_tools):
    return await run_agent(
        question=(
            f"Сгенерируй системный промпт для нового ассистента. Описание: {TARGET_ASSISTANT}. "
            "Опирайся на лучшие практики из сохранённого отчёта 'analysis_report'. "
            "Сохрани результат как 'generated_prompt'."
        ),
        session=session,
        mcp_tools=mcp_tools,
        skills_metadata=skills_metadata,
        max_steps=10,
        verbose=True,
    )


generator_trace = asyncio.get_event_loop().run_until_complete(with_github(stage_generator))

print(f"\n{'=' * 50}")
print(generator_trace.summary())
print(f"\n{'=' * 50}\nСгенерированный промпт:\n")
print(read_report("generated_prompt")[:3000])

[State] Задача: Сгенерируй системный промпт для нового ассистента. Описание: ассистент-помощник для студентов технического вуза: помогает разбирать учебные задачи по программированию, объясняет ошибки в коде и готови

--- Шаг 1 ---
  🔢 context=3878 (Δ +3878), completion=173
  🔧 read_skill({"name": "prompt_generator"})
  📡 ═══ SKILL ЗАГРУЖЕН: Генерация системного промпта ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Генерация системного промпта

## Требуемые ...
  🔧 read_report({"name": "analysis_report"})
  📡 # Сводный отчёт по системным промптам

## Обзор
Проанализировано промптов: **7**.  
Вендоры: **Anthropic** (100% — все промпты от Anthropic).  
Типы п...

--- Шаг 2 ---
  🔢 context=7860 (Δ +3982), completion=4189
  🔧 save_report({"name": "generated_prompt", "content": "# Системный промпт: Ассистент-наставник для студентов технического вуза\n\n## Д)
  📡 ✅ Отчёт сохранён: generated_prompt.md (10139 символов)

--- Шаг 3 ---
  🔢 context=9536 (Δ +1676), completion=396
  💬 #

<img src="pictures\hw_file_1.PNG" width="600"/>

<img src="pictures\hw_file_2.PNG" width="600"/>